In [1]:
import os
import requests
import subprocess

import geopandas as gpd
import pandas as pd

from pathlib import Path

In [ ]:

#### Downloading OSM data from Geofabrik

set_date = "250528" # 2025-05-28


#https://download.geofabrik.de/europe/germany/berlin-250401.osm.pbf
#	germany-250405.osm.pbf

def download_geofabrik_pbf(filename,base_url = "https://download.geofabrik.de/europe/"):
    folder_download = "osm_geofabrik_pbf"
    os.makedirs(folder_download, exist_ok=True)
    
    #filename = "germany-250401.osm.pbf"
    file_path = os.path.join(folder_download, filename)
    file_url = base_url + filename
    
    if os.path.exists(file_path):
        print(f"File already exists: {file_path}, skipping download.")
    else:
        print(f"Downloading: {file_url}")
        response = requests.get(file_url, stream=True, timeout=60)
        if response.status_code == 200:
            with open(file_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024):
                    f.write(chunk)
            print(f"Downloaded: {file_path}")
        else:
            print(f"Failed to download {file_url} (Status code: {response.status_code})")


# osmium needs to be installed on your system in order to run this code/filtering
# https://osmcode.org/osmium-tool/
# for my win11 machine i used https://trac.osgeo.org/osgeo4w/

def run_osmium(filename, outputfilename=None):
    try:
        folder_download = "osm_geofabrik_pbf"
        folder_processed = "processed_osm_files"
        os.makedirs(folder_processed, exist_ok=True)
        
        input_pbf = os.path.join(folder_download, filename)
        filtered_pbf = os.path.join(folder_processed, outputfilename)

        # # Convert to Unix-style paths using forward slashes
        # input_pbf = input_pbf.replace("\\", "/")
        # filtered_pbf = filtered_pbf.replace("\\", "/")

        if os.path.exists(filtered_pbf):
            print(f"Processed file already exists: {filtered_pbf}, skipping processing.")
            return

        filter_command = [
            "osmium", "tags-filter",
            input_pbf,
            #"n/amenity=school,kindergarten",  # filter nodes with school/kindergarten
            #"w/amenity=school,kindergarten",  # and ways, if any
            "nwr/amenity=school,kindergarten",
            "-o", filtered_pbf
        ]
        
        print("🔹 Running: ", " ".join(filter_command))
        subprocess.run(filter_command, check=True)

        print("✅ Osmium processing complete! Files saved in 'processed_osm_files/'")

    except subprocess.CalledProcessError as e:
        print("❌ Error running Osmium:", e)




filename = f"germany-{set_date}.osm.pbf"

download_geofabrik_pbf(filename )

run_osmium(filename,outputfilename=f"processed_schools_germany_{set_date}.pbf")


File already exists: osm_geofabrik_pbf/germany-250528.osm.pbf, skipping download.
🔹 Running:  osmium tags-filter osm_geofabrik_pbf/germany-250528.osm.pbf nwr/amenity=school,kindergarten -o processed_osm_files/processed_schools_germany_250528_rel.pbf
✅ Osmium processing complete! Files saved in 'processed_osm_files/'


In [13]:
import subprocess
import os

def ogr2ogr_to_geopackage(input_pbf, output_gpkg, osmconf_path=None):
    ogr2ogr_path = "ogr2ogr"

    cmd = [
        ogr2ogr_path,
        "-overwrite",                 # Overwrite the output file if it exists
        "--config",                   # Use configuration file
        "OSM_CONFIG_FILE",            # The config parameter key
        osmconf_path,                 # Path to the osmconf.ini
        "-f", "GPKG",  # Output format: GeoPackage
        #"-f", "Parquet",  # Output format: GeoPackage
        output_gpkg,   # Output GeoPackage file
        input_pbf#,      # Input .pbf file
       # "-oo OSM_CONFIG_FILE=C:/path/to/your/osmconf.ini"
    ]
    print("Running:", " ".join(cmd))  # Print the full command for debugging

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print("GeoPackage created successfully:")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("❌ ogr2ogr failed during conversion to GeoPackage:")
        print(e.stderr)
        raise

# Example usage
#input_pbf = "processed_osm_files/germany_osm_schools-25-05-09.pbf"
input_pbf = "processed_osm_files/processed_schools_germany_250528.pbf"

output_gpkg = "processed_osm_files/processed_schools_germany_250528.gpkg"
osmconf_path = "processed_osm_files/osmconf_schools.ini"

ogr2ogr_to_geopackage(input_pbf, output_gpkg, osmconf_path)

Running: ogr2ogr -overwrite --config OSM_CONFIG_FILE processed_osm_files/osmconf_schools.ini -f GPKG processed_osm_files/processed_schools_germany_250528.gpkg processed_osm_files/processed_schools_germany_250528.pbf
GeoPackage created successfully:
0...10...20...30...40...50...60...70...80...90...100 - done.



In [14]:
import geopandas as gpd
import fiona
import pandas as pd

def save_filtered_gpkg_layers_as_fgb(input_gpkg, output_fgb, selected_layers=None, allowed_amenities=None):
    # List all layers in the GeoPackage
    available_layers = fiona.listlayers(input_gpkg)
    print(f"📦 All available layers: {available_layers}")

    if selected_layers is None:
        selected_layers = available_layers

    # Filter to only existing layers
    layers_to_use = [layer for layer in selected_layers if layer in available_layers]
    print(f"✅ Layers to include: {layers_to_use}")

    all_gdfs = []

    for layer in layers_to_use:
        print(f"🔹 Reading layer: {layer}")
        gdf = gpd.read_file(input_gpkg, layer=layer)

        # Filter based on 'amenity'
        if 'amenity' in gdf.columns and allowed_amenities:
            before = len(gdf)
            gdf = gdf[gdf['amenity'].isin(allowed_amenities)]
            after = len(gdf)
            print(f"   🔸 Filtered {before} → {after} features by amenity")

        gdf["source_layer"] = layer  # optional
        all_gdfs.append(gdf)

    # Combine into one GeoDataFrame
    combined = pd.concat(all_gdfs, ignore_index=True)
    combined = gpd.GeoDataFrame(combined, geometry="geometry")

    # Write to FlatGeobuf
    combined.to_file(output_fgb, driver="FlatGeobuf")
    print(f"✅ FlatGeobuf file written: {output_fgb}")


In [15]:
save_filtered_gpkg_layers_as_fgb(
    input_gpkg="processed_osm_files/processed_schools_germany_250528.gpkg",
    output_fgb="processed_osm_files/processed_schools_germany_250528.fgb",
    selected_layers=["points", "multipolygons"],
    allowed_amenities=["school", "kindergarten"]
)

📦 All available layers: ['points', 'lines', 'multipolygons', 'other_relations', 'multilinestrings']
✅ Layers to include: ['points', 'multipolygons']
🔹 Reading layer: points
   🔸 Filtered 34267 → 18910 features by amenity
🔹 Reading layer: multipolygons
   🔸 Filtered 63282 → 62337 features by amenity
✅ FlatGeobuf file written: processed_osm_files/processed_schools_germany_250528.fgb


In [17]:
import subprocess

input_fgb = "processed_osm_files/processed_schools_germany_250528.fgb"

cmd = [
    "tippecanoe",
    "-o", "processed_osm_files/processed_schools_germany_250528.pmtiles",
    "--minimum-zoom=11",
    "--maximum-zoom=15",
    "--drop-rate=0",
    "--drop-densest-as-needed",
    "--no-feature-limit",
    "--no-tile-size-limit",
    "--maximum-tile-bytes=1000000",
    "--force",
    "-l", "germany_osm_schools",
    input_fgb
]

print("Running tippecanoe for schools...")
subprocess.run(cmd, check=True)


Running tippecanoe for schools...


detected indexed FlatGeobuf: assigning feature IDs by sequence
81247 features, 6820917 bytes of geometry and attributes, 2738406 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  15/17596/10752  


CompletedProcess(args=['tippecanoe', '-o', 'processed_osm_files/processed_schools_germany_250528.pmtiles', '--minimum-zoom=11', '--maximum-zoom=15', '--drop-rate=0', '--drop-densest-as-needed', '--no-feature-limit', '--no-tile-size-limit', '--maximum-tile-bytes=1000000', '--force', '-l', 'germany_osm_schools', 'processed_osm_files/processed_schools_germany_250528.fgb'], returncode=0)